In [2]:
from pathlib import Path

import duckdb
import pandas as pd
import numpy as np



In [3]:
PROJECT_ROOT = Path.cwd().parent
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"


con = duckdb.connect()

In [4]:
jan_path = INTERIM_DIR / "flights_2025_01.parquet"
print(jan_path.exists())
print(jan_path)


True
/Users/tseringgurung/Desktop/flight-operations-intelligence/data/interim/flights_2025_01.parquet


## Temporal Modeling Strategy

The model will be evaluated using chronological rather than random data splits.

Planned initial structure:

- Training: January–September 2025
- Validation: October–November 2025
- Test: December 2025

This prevents future observations from leaking into model training and better represents real operational deployment.

In [5]:
RAW_FLIGHT_DIR = PROJECT_ROOT / "data" / "raw" / "flights"

FEB_RAW_PATH = RAW_FLIGHT_DIR / "bts_ontime_2025_02.csv"

print(FEB_RAW_PATH.exists())
print(FEB_RAW_PATH)

True
/Users/tseringgurung/Desktop/flight-operations-intelligence/data/raw/flights/bts_ontime_2025_02.csv


In [6]:
con.execute(
    f"""
    CREATE OR REPLACE VIEW flights_feb_raw AS
    SELECT *
    FROM read_csv_auto(
        '{FEB_RAW_PATH}',
        sample_size = 100000,
        ignore_errors = true
    )
    """
)

In [7]:
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    MIN(FlightDate) AS first_date,
    MAX(FlightDate) AS last_date,
    COUNT(DISTINCT FlightDate) AS days
FROM flights_feb_raw
""").df()

,total_rows,first_date,last_date,days
0,504884,2025-02-01,2025-02-28,28


In [8]:
jan_schema = con.sql(
    f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{jan_path}')
    """
).df()

feb_schema = con.sql("""
DESCRIBE flights_feb_raw
""").df()

print("January columns:", len(jan_schema))
print("February columns:", len(feb_schema))

January columns: 110
February columns: 110


In [9]:
jan_columns = set(jan_schema["column_name"])
feb_columns = set(feb_schema["column_name"])

print("Missing in February:")
print(jan_columns - feb_columns)

print("\nNew in February:")
print(feb_columns - jan_columns)

Missing in February:
set()

New in February:
set()


In [10]:
same_order = (
    jan_schema["column_name"].tolist()
    == feb_schema["column_name"].tolist()
)

same_order

True

In [11]:
FEB_PARQUET_PATH = (
    PROJECT_ROOT 
    / "Data"
    / "INTERIM"
    /"flights_2025_02.parquet"
)

In [12]:
con.execute(
    f"""
    COPY (
        SELECT *
        FROM flights_feb_raw
    )
    TO '{FEB_PARQUET_PATH}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
    """
)

In [13]:
print(FEB_PARQUET_PATH.exists())

con.sql(
    f"""
    SELECT COUNT(*) AS total_rows
    FROM read_parquet('{FEB_PARQUET_PATH}')
    """
).df()

True


,total_rows
0,504884


In [14]:
feb_csv_size_mb = FEB_RAW_PATH.stat().st_size / 1024**2
feb_parquet_size_mb = FEB_PARQUET_PATH.stat().st_size / 1024**2

print(f"CSV size:     {feb_csv_size_mb:.2f} MB")
print(f"Parquet size: {feb_parquet_size_mb:.2f} MB")
print(
    f"Reduction:    "
    f"{(1 - feb_parquet_size_mb / feb_csv_size_mb) * 100:.1f}%"
)

CSV size:     217.67 MB
Parquet size: 12.06 MB
Reduction:    94.5%


In [15]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [16]:
from src.ingest_bts import process_bts_month

In [17]:
feb_result = process_bts_month(
    csv_path=FEB_RAW_PATH,
    parquet_path=FEB_PARQUET_PATH,
    expected_year=2025,
    expected_month=2,
)

feb_result

{'year': 2025,
 'month': 2,
 'rows': 504884,
 'first_date': Timestamp('2025-02-01 00:00:00'),
 'last_date': Timestamp('2025-02-28 00:00:00'),
 'days': 28,
 'csv_size_mb': 217.67,
 'parquet_size_mb': 12.06,
 'storage_reduction_pct': 94.5}

In [18]:
MAR_RAW_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "flights"
    / "bts_ontime_2025_03.csv"
)

MAR_PARQUET_PATH = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "flights_2025_03.parquet"
)

print(MAR_RAW_PATH.exists())

True


In [19]:
mar_result = process_bts_month(
    csv_path=MAR_RAW_PATH,
    parquet_path=MAR_PARQUET_PATH,
    expected_year=2025,
    expected_month=3,
)

mar_result

{'year': 2025,
 'month': 3,
 'rows': 600872,
 'first_date': Timestamp('2025-03-01 00:00:00'),
 'last_date': Timestamp('2025-03-31 00:00:00'),
 'days': 31,
 'csv_size_mb': 259.04,
 'parquet_size_mb': 14.58,
 'storage_reduction_pct': 94.4}

In [20]:
mar_result = process_bts_month(
    csv_path=MAR_RAW_PATH,
    parquet_path=MAR_PARQUET_PATH,
    expected_year=2025,
    expected_month=3,
    reference_parquet_path=jan_path,
)

mar_result

{'year': 2025,
 'month': 3,
 'rows': 600872,
 'first_date': Timestamp('2025-03-01 00:00:00'),
 'last_date': Timestamp('2025-03-31 00:00:00'),
 'days': 31,
 'csv_size_mb': 259.04,
 'parquet_size_mb': 14.58,
 'storage_reduction_pct': 94.4}

In [21]:
RAW_FLIGHT_DIR = PROJECT_ROOT / "data" / "raw" / "flights"

for month in range(1, 13):
    path = RAW_FLIGHT_DIR / f"bts_ontime_2025_{month:02d}.csv"
    print(f"{month:02d}: {path.exists()}")

01: True
02: True
03: True
04: True
05: True
06: True
07: True
08: True
09: True
10: True
11: True
12: True


In [22]:
ingestion_results = []

for month in range(4, 13):

    csv_path = (
        RAW_FLIGHT_DIR
        / f"bts_ontime_2025_{month:02d}.csv"
    )

    parquet_path = (
        INTERIM_DIR
        / f"flights_2025_{month:02d}.parquet"
    )

    print(f"Processing 2025-{month:02d}...")

    result = process_bts_month(
        csv_path=csv_path,
        parquet_path=parquet_path,
        expected_year=2025,
        expected_month=month,
        reference_parquet_path=jan_path,
    )

    ingestion_results.append(result)

    print(
        f"✓ {result['rows']:,} rows | "
        f"{result['storage_reduction_pct']}% reduction"
    )

Processing 2025-04...
✓ 583,950 rows | 94.5% reduction
Processing 2025-05...
✓ 605,648 rows | 94.5% reduction
Processing 2025-06...
✓ 611,575 rows | 94.5% reduction
Processing 2025-07...
✓ 631,428 rows | 94.4% reduction
Processing 2025-08...
✓ 602,378 rows | 94.4% reduction
Processing 2025-09...
✓ 562,439 rows | 94.6% reduction
Processing 2025-10...
✓ 605,844 rows | 94.5% reduction
Processing 2025-11...
✓ 570,550 rows | 94.4% reduction
Processing 2025-12...
✓ 582,304 rows | 94.1% reduction


In [23]:
ingestion_summary = pd.DataFrame(ingestion_results)

ingestion_summary

,year,month,rows,first_date,last_date,days,csv_size_mb,parquet_size_mb,storage_reduction_pct
0,2025,4,583950,2025-04-01,2025-04-30,30,251.85,13.96,94.5
1,2025,5,605648,2025-05-01,2025-05-31,31,261.62,14.33,94.5
2,2025,6,611575,2025-06-01,2025-06-30,30,264.55,14.64,94.5
3,2025,7,631428,2025-07-01,2025-07-31,31,272.92,15.21,94.4
4,2025,8,602378,2025-08-01,2025-08-31,31,260.06,14.49,94.4
5,2025,9,562439,2025-09-01,2025-09-30,30,242.30,13.14,94.6
6,2025,10,605844,2025-10-01,2025-10-31,31,261.98,14.42,94.5
7,2025,11,570550,2025-11-01,2025-11-30,30,246.12,13.84,94.4
8,2025,12,582304,2025-12-01,2025-12-31,31,252.27,14.90,94.1


In [24]:
all_flights_path = INTERIM_DIR / "flights_2025_*.parquet"

con.execute(
    f"""
    CREATE OR REPLACE VIEW flights_2025 AS
    SELECT *
    FROM read_parquet('{all_flights_path}')
    """
)

In [25]:
year_validation = con.sql("""
SELECT
    COUNT(*) AS total_rows,
    MIN(FlightDate) AS first_date,
    MAX(FlightDate) AS last_date,
    COUNT(DISTINCT FlightDate) AS days,
    COUNT(DISTINCT Month) AS months
FROM flights_2025
""").df()

year_validation

,total_rows,first_date,last_date,days,months
0,7001619,2025-01-01,2025-12-31,365,12


In [26]:
monthly_counts = con.sql("""
SELECT
    Month,
    COUNT(*) AS flights
FROM flights_2025
GROUP BY Month
ORDER BY Month
""").df()

monthly_counts

,Month,flights
0,1,539747
1,2,504884
2,3,600872
3,4,583950
4,5,605648
5,6,611575
6,7,631428
7,8,602378
8,9,562439
9,10,605844


In [27]:
modeling_2025 = con.sql("""
SELECT
    Year,
    Quarter,
    Month,
    DayofMonth,
    DayOfWeek,
    FlightDate,

    Reporting_Airline,
    Origin,
    Dest,

    CRSDepTime,
    CRSArrTime,
    CRSElapsedTime,

    Distance,
    DistanceGroup,

    Origin || '_' || Dest AS route,

    CAST(
        FLOOR(CAST(CRSDepTime AS INTEGER) / 100)
        AS INTEGER
    ) AS scheduled_dep_hour,

    CAST(
        FLOOR(CAST(CRSArrTime AS INTEGER) / 100)
        AS INTEGER
    ) AS scheduled_arr_hour,

    CASE
        WHEN DayOfWeek IN (6, 7) THEN 1
        ELSE 0
    END AS is_weekend,

    CASE
        WHEN CAST(CRSDepTime AS INTEGER) < 600 THEN 'overnight'
        WHEN CAST(CRSDepTime AS INTEGER) < 1200 THEN 'morning'
        WHEN CAST(CRSDepTime AS INTEGER) < 1800 THEN 'afternoon'
        ELSE 'evening'
    END AS departure_period,

    ArrDelay,

    CASE
        WHEN ArrDelay >= 15 THEN 1
        ELSE 0
    END AS significant_arrival_delay

FROM flights_2025

WHERE Cancelled = 0
  AND Diverted = 0
  AND ArrDelay IS NOT NULL
""")

In [28]:
modeling_2025.aggregate("""
    COUNT(*) AS modeling_rows
""").df()

,modeling_rows
0,6879484


In [29]:
year_target = con.sql("""
SELECT
    significant_arrival_delay,
    COUNT(*) AS flights,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS percentage
FROM modeling_2025
GROUP BY significant_arrival_delay
ORDER BY significant_arrival_delay
""").df()

year_target

,significant_arrival_delay,flights,percentage
0,0,5344846,77.69
1,1,1534638,22.31


In [30]:
monthly_target = con.sql("""
SELECT
    Month,
    COUNT(*) AS flights,
    SUM(significant_arrival_delay) AS delayed_flights,
    ROUND(
        AVG(significant_arrival_delay) * 100,
        2
    ) AS delay_rate_pct
FROM modeling_2025
GROUP BY Month
ORDER BY Month
""").df()

monthly_target

,Month,flights,delayed_flights,delay_rate_pct
0,1,522269,98130.0,18.79
1,2,496476,103102.0,20.77
2,3,592301,116034.0,19.59
3,4,577730,113604.0,19.66
4,5,597574,141038.0,23.60
5,6,599472,169405.0,28.26
6,7,612811,177012.0,28.89
7,8,593733,133920.0,22.56
8,9,558328,92859.0,16.63
9,10,601570,122251.0,20.32


In [31]:
train_2025 = con.sql("""
SELECT *
FROM modeling_2025
WHERE Month BETWEEN 1 AND 9
""")

validation_2025 = con.sql("""
SELECT *
FROM modeling_2025
WHERE Month BETWEEN 10 AND 11
""")

test_2025 = con.sql("""
SELECT *
FROM modeling_2025
WHERE Month = 12
""")

In [32]:
split_summary = con.sql("""
SELECT
    'train' AS split,
    COUNT(*) AS flights,
    MIN(FlightDate) AS first_date,
    MAX(FlightDate) AS last_date,
    ROUND(AVG(significant_arrival_delay) * 100, 2) AS delay_rate_pct
FROM train_2025

UNION ALL

SELECT
    'validation',
    COUNT(*),
    MIN(FlightDate),
    MAX(FlightDate),
    ROUND(AVG(significant_arrival_delay) * 100, 2)
FROM validation_2025

UNION ALL

SELECT
    'test',
    COUNT(*),
    MIN(FlightDate),
    MAX(FlightDate),
    ROUND(AVG(significant_arrival_delay) * 100, 2)
FROM test_2025
""").df()

split_summary

,split,flights,first_date,last_date,delay_rate_pct
0,train,5150694,2025-01-01,2025-09-30,22.23
1,validation,1156866,2025-10-01,2025-11-30,20.44
2,test,571924,2025-12-01,2025-12-31,26.77


In [34]:
carrier_daily = con.sql("""
SELECT
    FlightDate,
    Reporting_Airline,
    COUNT(*) AS daily_flights,
    SUM(significant_arrival_delay) AS daily_delayed_flights
FROM modeling_2025
GROUP BY
    FlightDate,
    Reporting_Airline
""")
carrier_daily


┌────────────┬───────────────────┬───────────────┬───────────────────────┐
│ FlightDate │ Reporting_Airline │ daily_flights │ daily_delayed_flights │
│    date    │      varchar      │     int64     │        int128         │
├────────────┼───────────────────┼───────────────┼───────────────────────┤
│ 2025-01-16 │ AA                │          2596 │                   371 │
│ 2025-01-26 │ AA                │          2547 │                   400 │
│ 2025-01-07 │ AS                │           484 │                   102 │
│ 2025-01-11 │ AS                │           481 │                    87 │
│ 2025-01-19 │ AS                │           611 │                   115 │
│ 2025-01-21 │ AS                │           488 │                    63 │
│ 2025-04-06 │ MQ                │           831 │                   213 │
│ 2025-04-11 │ MQ                │           840 │                   105 │
│ 2025-04-14 │ MQ                │           840 │                    82 │
│ 2025-04-06 │ OH        

In [35]:
carrier_history = con.sql("""
SELECT
    FlightDate,
    Reporting_Airline,

    SUM(daily_flights) OVER (
        PARTITION BY Reporting_Airline
        ORDER BY FlightDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS prior_carrier_flights,

    SUM(daily_delayed_flights) OVER (
        PARTITION BY Reporting_Airline
        ORDER BY FlightDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS prior_carrier_delays

FROM carrier_daily
""")

In [36]:
carrier_history_rates = con.sql("""
SELECT
    FlightDate,
    Reporting_Airline,
    prior_carrier_flights,
    prior_carrier_delays,

    CASE
        WHEN prior_carrier_flights > 0
        THEN prior_carrier_delays * 1.0 / prior_carrier_flights
        ELSE NULL
    END AS carrier_historical_delay_rate

FROM carrier_history
""")

In [38]:
con.sql("""
SELECT *
FROM carrier_history_rates
WHERE Reporting_Airline = 'AA'
ORDER BY FlightDate
LIMIT 10
""").df()

,FlightDate,Reporting_Airline,prior_carrier_flights,prior_carrier_delays,carrier_historical_delay_rate
0,2025-01-01,AA,NaN,NaN,NaN
1,2025-01-02,AA,2403.0,299.0,0.124428
2,2025-01-03,AA,5057.0,718.0,0.141981
3,2025-01-04,AA,7631.0,1264.0,0.165640
4,2025-01-05,AA,10001.0,1831.0,0.183082
5,2025-01-06,AA,12427.0,2812.0,0.226281
6,2025-01-07,AA,14778.0,3960.0,0.267966
7,2025-01-08,AA,16969.0,4319.0,0.254523
8,2025-01-09,AA,19195.0,4681.0,0.243866
9,2025-01-10,AA,21028.0,5106.0,0.242819


In [39]:
origin_daily = con.sql("""
SELECT
    FlightDate,
    Origin,
    COUNT(*) AS daily_flights,
    SUM(significant_arrival_delay) AS daily_delayed_flights
FROM modeling_2025
GROUP BY
    FlightDate,
    Origin
""")

In [40]:
origin_history = con.sql("""
SELECT
    FlightDate,
    Origin,

    SUM(daily_flights) OVER (
        PARTITION BY Origin
        ORDER BY FlightDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS prior_origin_flights,

    SUM(daily_delayed_flights) OVER (
        PARTITION BY Origin
        ORDER BY FlightDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS prior_origin_delays

FROM origin_daily
""")

In [41]:
origin_history_rates = con.sql("""
SELECT
    FlightDate,
    Origin,
    prior_origin_flights,
    prior_origin_delays,

    CASE
        WHEN prior_origin_flights > 0
        THEN prior_origin_delays * 1.0 / prior_origin_flights
        ELSE NULL
    END AS origin_historical_delay_rate

FROM origin_history
""")

In [42]:
con.sql("""
SELECT *
FROM origin_history_rates
WHERE Origin = 'ATL'
ORDER BY FlightDate
LIMIT 10
""").df()

,FlightDate,Origin,prior_origin_flights,prior_origin_delays,origin_historical_delay_rate
0,2025-01-01,ATL,NaN,NaN,NaN
1,2025-01-02,ATL,668.0,93.0,0.139222
2,2025-01-03,ATL,1461.0,249.0,0.170431
3,2025-01-04,ATL,2263.0,402.0,0.177640
4,2025-01-05,ATL,3023.0,647.0,0.214026
5,2025-01-06,ATL,3780.0,904.0,0.239153
6,2025-01-07,ATL,4511.0,1151.0,0.255154
7,2025-01-08,ATL,5213.0,1301.0,0.249568
8,2025-01-09,ATL,5929.0,1419.0,0.239332
9,2025-01-10,ATL,6662.0,1599.0,0.240018


In [43]:
destination_daily = con.sql("""
SELECT
    FlightDate,
    Dest,
    COUNT(*) AS daily_flights,
    SUM(significant_arrival_delay) AS daily_delayed_flights
FROM modeling_2025
GROUP BY
    FlightDate,
    Dest
""")

In [44]:
destination_history = con.sql("""
SELECT
    FlightDate,
    Dest,

    SUM(daily_flights) OVER (
        PARTITION BY Dest
        ORDER BY FlightDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS prior_destination_flights,

    SUM(daily_delayed_flights) OVER (
        PARTITION BY Dest
        ORDER BY FlightDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS prior_destination_delays

FROM destination_daily
""")

In [45]:
destination_history_rates = con.sql("""
SELECT
    FlightDate,
    Dest,
    prior_destination_flights,
    prior_destination_delays,

    CASE
        WHEN prior_destination_flights > 0
        THEN prior_destination_delays * 1.0 / prior_destination_flights
        ELSE NULL
    END AS destination_historical_delay_rate

FROM destination_history
""")

In [46]:
con.sql("""
SELECT *
FROM destination_history_rates
WHERE Dest = 'ATL'
ORDER BY FlightDate
LIMIT 10
""").df()

,FlightDate,Dest,prior_destination_flights,prior_destination_delays,destination_historical_delay_rate
0,2025-01-01,ATL,NaN,NaN,NaN
1,2025-01-02,ATL,663.0,76.0,0.114630
2,2025-01-03,ATL,1465.0,195.0,0.133106
3,2025-01-04,ATL,2265.0,335.0,0.147903
4,2025-01-05,ATL,3031.0,526.0,0.173540
5,2025-01-06,ATL,3806.0,772.0,0.202838
6,2025-01-07,ATL,4515.0,1003.0,0.222148
7,2025-01-08,ATL,5213.0,1121.0,0.215039
8,2025-01-09,ATL,5926.0,1221.0,0.206041
9,2025-01-10,ATL,6635.0,1327.0,0.200000


In [47]:
route_daily = con.sql("""
SELECT
    FlightDate,
    route,
    COUNT(*) AS daily_flights,
    SUM(significant_arrival_delay) AS daily_delayed_flights
FROM modeling_2025
GROUP BY
    FlightDate,
    route
""")

In [48]:
route_history = con.sql("""
SELECT
    FlightDate,
    route,

    SUM(daily_flights) OVER (
        PARTITION BY route
        ORDER BY FlightDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS prior_route_flights,

    SUM(daily_delayed_flights) OVER (
        PARTITION BY route
        ORDER BY FlightDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS prior_route_delays

FROM route_daily
""")

In [49]:
route_history_rates = con.sql("""
SELECT
    FlightDate,
    route,
    prior_route_flights,
    prior_route_delays,

    CASE
        WHEN prior_route_flights > 0
        THEN prior_route_delays * 1.0 / prior_route_flights
        ELSE NULL
    END AS route_historical_delay_rate

FROM route_history
""")

In [50]:
con.sql("""
SELECT *
FROM route_history_rates
WHERE route = 'ATL_MCO'
ORDER BY FlightDate
LIMIT 10
""").df()

,FlightDate,route,prior_route_flights,prior_route_delays,route_historical_delay_rate
0,2025-01-01,ATL_MCO,NaN,NaN,NaN
1,2025-01-02,ATL_MCO,19.0,1.0,0.052632
2,2025-01-03,ATL_MCO,42.0,4.0,0.095238
3,2025-01-04,ATL_MCO,66.0,8.0,0.121212
4,2025-01-05,ATL_MCO,89.0,18.0,0.202247
5,2025-01-06,ATL_MCO,114.0,25.0,0.219298
6,2025-01-07,ATL_MCO,135.0,31.0,0.229630
7,2025-01-08,ATL_MCO,155.0,37.0,0.238710
8,2025-01-09,ATL_MCO,175.0,44.0,0.251429
9,2025-01-10,ATL_MCO,197.0,48.0,0.243655


In [51]:
modeling_with_history = con.sql("""
SELECT
    m.*,

    c.prior_carrier_flights,
    c.carrier_historical_delay_rate,

    o.prior_origin_flights,
    o.origin_historical_delay_rate,

    d.prior_destination_flights,
    d.destination_historical_delay_rate,

    r.prior_route_flights,
    r.route_historical_delay_rate

FROM modeling_2025 m

LEFT JOIN carrier_history_rates c
    ON m.FlightDate = c.FlightDate
   AND m.Reporting_Airline = c.Reporting_Airline

LEFT JOIN origin_history_rates o
    ON m.FlightDate = o.FlightDate
   AND m.Origin = o.Origin

LEFT JOIN destination_history_rates d
    ON m.FlightDate = d.FlightDate
   AND m.Dest = d.Dest

LEFT JOIN route_history_rates r
    ON m.FlightDate = r.FlightDate
   AND m.route = r.route
""")

In [52]:
con.sql("""
SELECT
    (SELECT COUNT(*) FROM modeling_2025) AS original_rows,
    (SELECT COUNT(*) FROM modeling_with_history) AS joined_rows
""").df()

,original_rows,joined_rows
0,6879484,6879484


In [53]:
history_missingness = con.sql("""
SELECT
    COUNT(*) AS total_rows,

    SUM(CASE WHEN carrier_historical_delay_rate IS NULL THEN 1 ELSE 0 END)
        AS missing_carrier_history,

    SUM(CASE WHEN origin_historical_delay_rate IS NULL THEN 1 ELSE 0 END)
        AS missing_origin_history,

    SUM(CASE WHEN destination_historical_delay_rate IS NULL THEN 1 ELSE 0 END)
        AS missing_destination_history,

    SUM(CASE WHEN route_historical_delay_rate IS NULL THEN 1 ELSE 0 END)
        AS missing_route_history

FROM modeling_with_history
""").df()

history_missingness

,total_rows,missing_carrier_history,missing_origin_history,missing_destination_history,missing_route_history
0,6879484,16771.0,16831.0,16833.0,19155.0


In [54]:
con.sql("""
SELECT
    MIN(prior_route_flights) AS min_history,
    PERCENTILE_CONT(0.10) WITHIN GROUP (ORDER BY prior_route_flights) AS p10,
    PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY prior_route_flights) AS p25,
    PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY prior_route_flights) AS median,
    PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY prior_route_flights) AS p75,
    PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY prior_route_flights) AS p90,
    MAX(prior_route_flights) AS max_history
FROM modeling_with_history
WHERE prior_route_flights IS NOT NULL
""").df()

,min_history,p10,p25,median,p75,p90,max_history
0,1.0,116.0,333.0,862.0,1860.0,3317.0,11187.0


In [55]:
system_daily = con.sql("""
SELECT
    FlightDate,
    COUNT(*) AS daily_flights,
    SUM(significant_arrival_delay) AS daily_delayed_flights
FROM modeling_2025
GROUP BY FlightDate
""")

In [56]:
system_history = con.sql("""
SELECT
    FlightDate,

    SUM(daily_flights) OVER (
        ORDER BY FlightDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS prior_system_flights,

    SUM(daily_delayed_flights) OVER (
        ORDER BY FlightDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS prior_system_delays

FROM system_daily
""")

In [57]:
system_history_rates = con.sql("""
SELECT
    FlightDate,
    prior_system_flights,
    prior_system_delays,

    CASE
        WHEN prior_system_flights > 0
        THEN prior_system_delays * 1.0 / prior_system_flights
        ELSE NULL
    END AS system_historical_delay_rate

FROM system_history
""")

In [58]:
con.sql("""
SELECT *
FROM system_history_rates
ORDER BY FlightDate
LIMIT 10
""").df()

,FlightDate,prior_system_flights,prior_system_delays,system_historical_delay_rate
0,2025-01-01,NaN,NaN,NaN
1,2025-01-02,16771.0,2566.0,0.153002
2,2025-01-03,36202.0,5851.0,0.161621
3,2025-01-04,55430.0,11016.0,0.198737
4,2025-01-05,73687.0,16474.0,0.223567
5,2025-01-06,91838.0,22488.0,0.244866
6,2025-01-07,108654.0,29240.0,0.269111
7,2025-01-08,124099.0,33117.0,0.266860
8,2025-01-09,140170.0,35490.0,0.253193
9,2025-01-10,156107.0,39323.0,0.251898


In [59]:
system_daily = con.sql("""
SELECT
    FlightDate,
    COUNT(*) AS daily_flights,
    SUM(significant_arrival_delay) AS daily_delayed_flights
FROM modeling_2025
GROUP BY FlightDate
""")

In [60]:
system_history = con.sql("""
SELECT
    FlightDate,

    SUM(daily_flights) OVER (
        ORDER BY FlightDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS prior_system_flights,

    SUM(daily_delayed_flights) OVER (
        ORDER BY FlightDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS prior_system_delays

FROM system_daily
""")

In [61]:
system_history_rates = con.sql("""
SELECT
    FlightDate,
    prior_system_flights,
    prior_system_delays,

    CASE
        WHEN prior_system_flights > 0
        THEN prior_system_delays * 1.0 / prior_system_flights
        ELSE NULL
    END AS system_historical_delay_rate

FROM system_history
""")

In [62]:
con.sql("""
SELECT *
FROM system_history_rates
ORDER BY FlightDate
LIMIT 10
""").df()

,FlightDate,prior_system_flights,prior_system_delays,system_historical_delay_rate
0,2025-01-01,NaN,NaN,NaN
1,2025-01-02,16771.0,2566.0,0.153002
2,2025-01-03,36202.0,5851.0,0.161621
3,2025-01-04,55430.0,11016.0,0.198737
4,2025-01-05,73687.0,16474.0,0.223567
5,2025-01-06,91838.0,22488.0,0.244866
6,2025-01-07,108654.0,29240.0,0.269111
7,2025-01-08,124099.0,33117.0,0.266860
8,2025-01-09,140170.0,35490.0,0.253193
9,2025-01-10,156107.0,39323.0,0.251898


In [63]:
modeling_with_all_history = con.sql("""
SELECT
    m.*,
    s.prior_system_flights,
    s.system_historical_delay_rate

FROM modeling_with_history m

LEFT JOIN system_history_rates s
    ON m.FlightDate = s.FlightDate
""")

In [64]:
con.sql("""
SELECT
    (SELECT COUNT(*) FROM modeling_with_history) AS before_join,
    (SELECT COUNT(*) FROM modeling_with_all_history) AS after_join
""").df()

,before_join,after_join
0,6879484,6879484


In [65]:
modeling_final_features = con.sql("""
SELECT
    *,

    COALESCE(
        carrier_historical_delay_rate,
        system_historical_delay_rate
    ) AS carrier_delay_rate_final,

    COALESCE(
        origin_historical_delay_rate,
        system_historical_delay_rate
    ) AS origin_delay_rate_final,

    COALESCE(
        destination_historical_delay_rate,
        system_historical_delay_rate
    ) AS destination_delay_rate_final,

    COALESCE(
        route_historical_delay_rate,
        origin_historical_delay_rate,
        destination_historical_delay_rate,
        carrier_historical_delay_rate,
        system_historical_delay_rate
    ) AS route_delay_rate_final,

    CASE WHEN carrier_historical_delay_rate IS NULL THEN 1 ELSE 0 END
        AS carrier_history_missing,

    CASE WHEN origin_historical_delay_rate IS NULL THEN 1 ELSE 0 END
        AS origin_history_missing,

    CASE WHEN destination_historical_delay_rate IS NULL THEN 1 ELSE 0 END
        AS destination_history_missing,

    CASE WHEN route_historical_delay_rate IS NULL THEN 1 ELSE 0 END
        AS route_history_missing

FROM modeling_with_all_history
""")

In [66]:
final_history_missingness = con.sql("""
SELECT
    COUNT(*) AS total_rows,

    SUM(CASE WHEN carrier_delay_rate_final IS NULL THEN 1 ELSE 0 END)
        AS missing_carrier_final,

    SUM(CASE WHEN origin_delay_rate_final IS NULL THEN 1 ELSE 0 END)
        AS missing_origin_final,

    SUM(CASE WHEN destination_delay_rate_final IS NULL THEN 1 ELSE 0 END)
        AS missing_destination_final,

    SUM(CASE WHEN route_delay_rate_final IS NULL THEN 1 ELSE 0 END)
        AS missing_route_final

FROM modeling_final_features
""").df()

final_history_missingness

,total_rows,missing_carrier_final,missing_origin_final,missing_destination_final,missing_route_final
0,6879484,16771.0,16771.0,16771.0,16771.0


In [71]:
modeling_ready = con.sql("""
SELECT *
FROM modeling_final_features
WHERE system_historical_delay_rate IS NOT NULL
""")

In [72]:
con.sql("""
SELECT
    COUNT(*) AS modeling_rows,
    MIN(FlightDate) AS first_date,
    MAX(FlightDate) AS last_date,
    COUNT(DISTINCT FlightDate) AS days
FROM modeling_ready
""").df()

,modeling_rows,first_date,last_date,days
0,6862713,2025-01-02,2025-12-31,364


In [73]:
baseline_features = [
    # Schedule / calendar
    "Month",
    "DayOfWeek",
    "scheduled_dep_hour",
    "scheduled_arr_hour",
    "is_weekend",

    # Flight characteristics
    "Reporting_Airline",
    "Origin",
    "Dest",
    "route",
    "CRSElapsedTime",
    "Distance",

    # Leakage-safe historical performance
    "carrier_delay_rate_final",
    "origin_delay_rate_final",
    "destination_delay_rate_final",
    "route_delay_rate_final",

    # Amount of historical evidence
    "prior_carrier_flights",
    "prior_origin_flights",
    "prior_destination_flights",
    "prior_route_flights",

    # Cold-start indicators
    "carrier_history_missing",
    "origin_history_missing",
    "destination_history_missing",
    "route_history_missing"
]

target = "significant_arrival_delay"

In [74]:
leakage_columns = {
    "DepTime",
    "DepDelay",
    "DepDelayMinutes",
    "DepDel15",
    "DepartureDelayGroups",
    "TaxiOut",
    "WheelsOff",
    "WheelsOn",
    "TaxiIn",
    "ArrTime",
    "ArrDelay",
    "ArrDelayMinutes",
    "ArrDel15",
    "ArrivalDelayGroups",
    "ActualElapsedTime",
    "AirTime",
    "CarrierDelay",
    "WeatherDelay",
    "NASDelay",
    "SecurityDelay",
    "LateAircraftDelay"
}

leaked_features = set(baseline_features) & leakage_columns

print("Leakage columns found:", leaked_features)

Leakage columns found: set()


In [75]:
available_columns = set(
    modeling_ready.columns
)

missing_features = [
    col
    for col in baseline_features
    if col not in available_columns
]

print("Missing features:", missing_features)

Missing features: []


In [76]:
train_final = con.sql("""
SELECT *
FROM modeling_ready
WHERE Month BETWEEN 1 AND 9
""")

validation_final = con.sql("""
SELECT *
FROM modeling_ready
WHERE Month BETWEEN 10 AND 11
""")

test_final = con.sql("""
SELECT *
FROM modeling_ready
WHERE Month = 12
""")

In [77]:
final_split_summary = con.sql("""
SELECT
    'train' AS split,
    COUNT(*) AS rows,
    MIN(FlightDate) AS first_date,
    MAX(FlightDate) AS last_date,
    ROUND(AVG(significant_arrival_delay) * 100, 2) AS delay_rate_pct
FROM train_final

UNION ALL

SELECT
    'validation',
    COUNT(*),
    MIN(FlightDate),
    MAX(FlightDate),
    ROUND(AVG(significant_arrival_delay) * 100, 2)
FROM validation_final

UNION ALL

SELECT
    'test',
    COUNT(*),
    MIN(FlightDate),
    MAX(FlightDate),
    ROUND(AVG(significant_arrival_delay) * 100, 2)
FROM test_final
""").df()

final_split_summary

,split,rows,first_date,last_date,delay_rate_pct
0,train,5133923,2025-01-02,2025-09-30,22.25
1,validation,1156866,2025-10-01,2025-11-30,20.44
2,test,571924,2025-12-01,2025-12-31,26.77


In [78]:
categorical_features = [
    "Reporting_Airline",
    "Origin",
    "Dest"
]

numeric_features = [
    "Month",
    "DayOfWeek",
    "scheduled_dep_hour",
    "scheduled_arr_hour",
    "is_weekend",
    "CRSElapsedTime",
    "Distance",

    "carrier_delay_rate_final",
    "origin_delay_rate_final",
    "destination_delay_rate_final",
    "route_delay_rate_final",

    "prior_carrier_flights",
    "prior_origin_flights",
    "prior_destination_flights",
    "prior_route_flights",

    "carrier_history_missing",
    "origin_history_missing",
    "destination_history_missing",
    "route_history_missing"
]

baseline_v1_features = (
    categorical_features
    + numeric_features
)

len(baseline_v1_features)

22

In [79]:
null_check = con.sql("""
SELECT
    SUM(CASE WHEN CRSElapsedTime IS NULL THEN 1 ELSE 0 END)
        AS missing_crs_elapsed,

    SUM(CASE WHEN Distance IS NULL THEN 1 ELSE 0 END)
        AS missing_distance,

    SUM(CASE WHEN scheduled_dep_hour IS NULL THEN 1 ELSE 0 END)
        AS missing_dep_hour,

    SUM(CASE WHEN scheduled_arr_hour IS NULL THEN 1 ELSE 0 END)
        AS missing_arr_hour,

    SUM(CASE WHEN prior_carrier_flights IS NULL THEN 1 ELSE 0 END)
        AS missing_prior_carrier,

    SUM(CASE WHEN prior_origin_flights IS NULL THEN 1 ELSE 0 END)
        AS missing_prior_origin,

    SUM(CASE WHEN prior_destination_flights IS NULL THEN 1 ELSE 0 END)
        AS missing_prior_destination,

    SUM(CASE WHEN prior_route_flights IS NULL THEN 1 ELSE 0 END)
        AS missing_prior_route

FROM modeling_ready
""").df()

null_check

,missing_crs_elapsed,missing_distance,missing_dep_hour,missing_arr_hour,missing_prior_carrier,missing_prior_origin,missing_prior_destination,missing_prior_route
0,0.0,0.0,0.0,0.0,0.0,60.0,62.0,2384.0


In [80]:
ml_ready = con.sql("""
SELECT
    *,
    
    COALESCE(prior_carrier_flights, 0) AS prior_carrier_flights_ml,
    COALESCE(prior_origin_flights, 0) AS prior_origin_flights_ml,
    COALESCE(prior_destination_flights, 0) AS prior_destination_flights_ml,
    COALESCE(prior_route_flights, 0) AS prior_route_flights_ml

FROM modeling_ready
""")

In [81]:
numeric_features = [
    "Month",
    "DayOfWeek",
    "scheduled_dep_hour",
    "scheduled_arr_hour",
    "is_weekend",
    "CRSElapsedTime",
    "Distance",

    "carrier_delay_rate_final",
    "origin_delay_rate_final",
    "destination_delay_rate_final",
    "route_delay_rate_final",

    "prior_carrier_flights_ml",
    "prior_origin_flights_ml",
    "prior_destination_flights_ml",
    "prior_route_flights_ml",

    "carrier_history_missing",
    "origin_history_missing",
    "destination_history_missing",
    "route_history_missing"
]

baseline_v1_features = categorical_features + numeric_features

print("Baseline features:", len(baseline_v1_features))

Baseline features: 22


In [82]:
null_counts = con.sql("""
SELECT
    COUNT(*) AS rows_with_any_null
FROM ml_ready
WHERE
    Reporting_Airline IS NULL
    OR Origin IS NULL
    OR Dest IS NULL
    OR Month IS NULL
    OR DayOfWeek IS NULL
    OR scheduled_dep_hour IS NULL
    OR scheduled_arr_hour IS NULL
    OR is_weekend IS NULL
    OR CRSElapsedTime IS NULL
    OR Distance IS NULL
    OR carrier_delay_rate_final IS NULL
    OR origin_delay_rate_final IS NULL
    OR destination_delay_rate_final IS NULL
    OR route_delay_rate_final IS NULL
    OR prior_carrier_flights_ml IS NULL
    OR prior_origin_flights_ml IS NULL
    OR prior_destination_flights_ml IS NULL
    OR prior_route_flights_ml IS NULL
    OR carrier_history_missing IS NULL
    OR origin_history_missing IS NULL
    OR destination_history_missing IS NULL
    OR route_history_missing IS NULL
    OR significant_arrival_delay IS NULL
""").df()

null_counts

,rows_with_any_null
0,0


In [83]:
feature_sql = ", ".join(baseline_v1_features)

train_ml = con.sql(f"""
SELECT
    {feature_sql},
    significant_arrival_delay
FROM ml_ready
WHERE Month BETWEEN 1 AND 9
""")

validation_ml = con.sql(f"""
SELECT
    {feature_sql},
    significant_arrival_delay
FROM ml_ready
WHERE Month BETWEEN 10 AND 11
""")

test_ml = con.sql(f"""
SELECT
    {feature_sql},
    significant_arrival_delay
FROM ml_ready
WHERE Month = 12
""")

In [84]:
con.sql("""
SELECT 'train' AS split, COUNT(*) AS rows FROM train_ml

UNION ALL

SELECT 'validation', COUNT(*) FROM validation_ml

UNION ALL

SELECT 'test', COUNT(*) FROM test_ml
""").df()

,split,rows
0,train,5133923
1,validation,1156866
2,test,571924


In [85]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [86]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            ),
            categorical_features
        ),
        (
            "numeric",
            StandardScaler(),
            numeric_features
        )
    ]
)

In [87]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

In [101]:
baseline_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                tol=1e-3,
                class_weight="balanced",
                solver="saga",
                n_jobs=-1
            )
        )
    ]
)

In [89]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

In [90]:
train_sample = train_ml.limit(100_000).df()

sample_memory_mb = (
    train_sample.memory_usage(deep=True).sum() / 1024**2
)

sample_memory_mb

np.float64(29.277923583984375)

In [92]:
import psutil

memory = psutil.virtual_memory()

print(f"Total RAM: {memory.total / 1024**3:.1f} GB")
print(f"Available RAM: {memory.available / 1024**3:.1f} GB")
print(f"Used RAM: {memory.used / 1024**3:.1f} GB")

Total RAM: 24.0 GB
Available RAM: 9.5 GB
Used RAM: 11.5 GB


### Baseline Modeling Strategy

With approximately **9.5 GB of available memory**, fitting the full **5.13 million-row training dataset** with logistic regression would introduce unnecessary memory risk. Although it may be technically possible, pandas conversion, one-hot encoding, sparse matrix construction, and model fitting all add additional memory overhead.

For the initial baseline, we will therefore use a **1 million-row training sample**.

This sample is sufficiently large to establish a robust baseline while:

- Reducing memory pressure
- Shortening model training time
- Allowing faster experimentation and debugging
- Avoiding unnecessary hardware constraints during baseline development

The full training dataset can be revisited later if model performance or validation results indicate that additional training observations are likely to provide meaningful value.

In [93]:
train_sample_ml = con.sql(f"""
SELECT
    {feature_sql},
    significant_arrival_delay
FROM ml_ready
WHERE Month BETWEEN 1 AND 9
USING SAMPLE 1000000 ROWS (reservoir, 42)
""")

In [94]:
con.sql("""
SELECT
    'full_train' AS dataset,
    COUNT(*) AS rows,
    ROUND(AVG(significant_arrival_delay) * 100, 2) AS delay_rate_pct
FROM train_ml

UNION ALL

SELECT
    'sample_train',
    COUNT(*),
    ROUND(AVG(significant_arrival_delay) * 100, 2)
FROM train_sample_ml
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,dataset,rows,delay_rate_pct
0,full_train,5133923,22.25
1,sample_train,748229,22.30


In [95]:
train_sample_ml = con.sql(f"""
SELECT
    {feature_sql},
    significant_arrival_delay
FROM ml_ready
WHERE Month BETWEEN 1 AND 9
ORDER BY RANDOM()
LIMIT 1000000
""")

In [96]:
train_sample_ml.aggregate("""
    COUNT(*) AS rows
""").df()

,rows
0,1000000


In [97]:
con.sql("""
SELECT
    'full_train' AS dataset,
    COUNT(*) AS rows,
    ROUND(AVG(significant_arrival_delay) * 100, 2) AS delay_rate_pct
FROM train_ml

UNION ALL

SELECT
    'sample_train',
    COUNT(*),
    ROUND(AVG(significant_arrival_delay) * 100, 2)
FROM train_sample_ml
""").df()

,dataset,rows,delay_rate_pct
0,full_train,5133923,22.25
1,sample_train,1000000,22.26


In [98]:
train_df = train_sample_ml.df()

X_train = train_df[baseline_v1_features]
y_train = train_df["significant_arrival_delay"]

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

X_train shape: (1000000, 22)
y_train shape: (1000000,)


In [102]:
baseline_model.fit(X_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Reporting_Airline',
                                                   'Origin', 'Dest']),
                                                 ('numeric', StandardScaler(),
                                                  ['Month', 'DayOfWeek',
                                                   'scheduled_dep_hour',
                                                   'scheduled_arr_hour',
                                                   'is_weekend',
                                                   'CRSElapsedTime', 'Distance',
                                                   'carrier_delay_rate_final',
                                                   'origin_delay_rate_final',
                                                   'destina...
                                                   'route_delay_rate_final',
                                                   'prior_carrier_flights_ml',
                                                   'prior_origin_flights_ml',
                                                   'prior_destination_flights_ml',
                                                   'prior_route_flights_ml',
                                                   'carrier_history_missing',
                                                   'origin_history_missing',
                                                   'destination_history_missing',
                                                   'route_history_missing'])])),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    n_jobs=-1, solver='saga', tol=0.001))])

In [103]:
classifier = baseline_model.named_steps["classifier"]

print("Iterations used:", classifier.n_iter_)

Iterations used: [506]


# Loading Validation Data

In [105]:
validation_df = validation_ml.df()

X_validation = validation_df[baseline_v1_features]
y_validation = validation_df["significant_arrival_delay"]

print("X_validation shape:", X_validation.shape)
print("y_validation shape:", y_validation.shape)

X_validation shape: (1156866, 22)
y_validation shape: (1156866,)


In [106]:
y_val_pred = baseline_model.predict(X_validation)

y_val_prob = baseline_model.predict_proba(X_validation)[:, 1]

In [107]:
validation_metrics = {
    "accuracy": accuracy_score(y_validation, y_val_pred),
    "precision": precision_score(y_validation, y_val_pred),
    "recall": recall_score(y_validation, y_val_pred),
    "f1": f1_score(y_validation, y_val_pred),
    "roc_auc": roc_auc_score(y_validation, y_val_prob),
    "pr_auc": average_precision_score(y_validation, y_val_prob)
}

validation_metrics

{'accuracy': 0.5795390304495075,
 'precision': 0.27665208628157295,
 'recall': 0.6548910629766391,
 'f1': 0.3889824728009979,
 'roc_auc': np.float64(0.6445144272652213),
 'pr_auc': np.float64(0.29866581536731945)}

In [108]:
confusion_matrix(y_validation, y_val_pred)

array([[515619, 404826],
       [ 81591, 154830]])

In [109]:
import numpy as np

y_naive = np.zeros(len(y_validation), dtype=int)

naive_metrics = {
    "accuracy": accuracy_score(y_validation, y_naive),
    "precision": precision_score(
        y_validation,
        y_naive,
        zero_division=0
    ),
    "recall": recall_score(y_validation, y_naive),
    "f1": f1_score(y_validation, y_naive)
}

naive_metrics

{'accuracy': 0.7956366597341438, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}

In [111]:
import pandas as pd

threshold_results = []

for threshold in [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]:
    
    predictions = (y_val_prob >= threshold).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(
            y_validation, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_validation, predictions
        ),
        "f1": f1_score(
            y_validation, predictions
        ),
        "accuracy": accuracy_score(
            y_validation, predictions
        )
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_df.round(3)

,threshold,precision,recall,f1,accuracy
0,0.30,0.216,0.964,0.353,0.279
1,0.35,0.228,0.918,0.365,0.348
2,0.40,0.242,0.850,0.377,0.427
3,0.45,0.259,0.761,0.387,0.506
4,0.50,0.277,0.655,0.389,0.580
5,0.55,0.295,0.530,0.379,0.645
6,0.60,0.315,0.393,0.349,0.701
7,0.65,0.334,0.247,0.284,0.745
8,0.70,0.357,0.123,0.183,0.776


In [112]:
baseline_results = pd.DataFrame({
    "model": ["Logistic Regression v1"],
    "training_rows": [1_000_000],
    "threshold": [0.50],
    "accuracy": [accuracy_score(y_validation, y_val_pred)],
    "precision": [precision_score(y_validation, y_val_pred)],
    "recall": [recall_score(y_validation, y_val_pred)],
    "f1": [f1_score(y_validation, y_val_pred)],
    "roc_auc": [roc_auc_score(y_validation, y_val_prob)],
    "pr_auc": [average_precision_score(y_validation, y_val_prob)]
})

baseline_results.round(3)

,model,training_rows,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc
0,Logistic Regression v1,1000000,0.5,0.58,0.277,0.655,0.389,0.645,0.299


## Baseline Model Evaluation

The first predictive benchmark uses logistic regression trained on a reproducible 1,000,000-row sample from the January–September 2025 training period. The model uses only information available before scheduled departure, including schedule characteristics, airline and airport identifiers, and leakage-safe historical delay features.

The model was evaluated on the temporally held-out October–November validation period.

### Validation Results

| Metric | Score |
|---|---:|
| Accuracy | 0.580 |
| Precision | 0.277 |
| Recall | 0.655 |
| F1 Score | 0.389 |
| ROC-AUC | 0.645 |
| PR-AUC | 0.299 |

A naive classifier that predicts no significant delay for every flight achieves approximately 79.6% accuracy but has zero recall for delayed flights. This demonstrates why accuracy alone is inappropriate for evaluating this imbalanced classification problem.

Threshold analysis showed the expected precision–recall tradeoff, but no tested threshold materially improved overall discrimination. The baseline therefore establishes that scheduled-flight and historical-performance features contain predictive signal, while also indicating that additional operational features are needed.

### Next Step

The next model iteration will incorporate leakage-safe rolling recent-history features to capture short-term changes in carrier, airport, route, and system disruption conditions. The December 2025 test period remains untouched as the final holdout set.

## Baseline v2 — Rolling Recent-History Features

The first logistic regression baseline demonstrated moderate predictive signal from scheduled-flight characteristics and cumulative historical delay rates. However, cumulative rates can respond slowly to changing operational conditions because older observations continue to influence the feature throughout the year.

To capture short-term disruption patterns, the second feature iteration introduces rolling historical delay rates based only on flights occurring during the previous 7 days.

Rolling features are constructed for:

- the overall flight system,
- reporting carrier,
- origin airport,
- destination airport, and
- origin–destination route.

For a flight on date *t*, the rolling window includes information from *t − 7* through *t − 1*. The current day's outcomes are explicitly excluded to prevent target leakage.

These features are intended to capture temporary operational regimes such as periods of elevated congestion, network disruption, or persistent airport and carrier performance deterioration.

In [113]:
system_daily = con.sql("""
SELECT
    FlightDate,
    COUNT(*) AS daily_system_flights,
    SUM(significant_arrival_delay) AS daily_system_delays
FROM modeling_2025
GROUP BY FlightDate
ORDER BY FlightDate
""")

In [114]:
system_7d_history = con.sql("""
SELECT
    FlightDate,

    SUM(daily_system_flights) OVER (
        ORDER BY FlightDate
        RANGE BETWEEN INTERVAL 7 DAY PRECEDING
                  AND INTERVAL 1 DAY PRECEDING
    ) AS prior_7d_system_flights,

    SUM(daily_system_delays) OVER (
        ORDER BY FlightDate
        RANGE BETWEEN INTERVAL 7 DAY PRECEDING
                  AND INTERVAL 1 DAY PRECEDING
    ) AS prior_7d_system_delays

FROM system_daily
ORDER BY FlightDate
""")

In [118]:
system_7d_history = con.sql("""
SELECT
    FlightDate,

    SUM(daily_system_flights) OVER (
        ORDER BY FlightDate
        RANGE BETWEEN INTERVAL 7 DAY PRECEDING
                  AND INTERVAL 1 DAY PRECEDING
    ) AS prior_7d_system_flights,

    SUM(daily_system_delays) OVER (
        ORDER BY FlightDate
        RANGE BETWEEN INTERVAL 7 DAY PRECEDING
                  AND INTERVAL 1 DAY PRECEDING
    ) AS prior_7d_system_delays

FROM system_daily
ORDER BY FlightDate
""")

In [121]:
system_7d_rates = con.sql("""
SELECT
    *,
    prior_7d_system_delays * 1.0
        / NULLIF(prior_7d_system_flights, 0)
        AS system_delay_rate_7d
FROM system_7d_history
""")

In [122]:
system_7d_rates.limit(12).df()

,FlightDate,prior_7d_system_flights,prior_7d_system_delays,system_delay_rate_7d
0,2025-01-01,NaN,NaN,NaN
1,2025-01-02,16771.0,2566.0,0.153002
2,2025-01-03,36202.0,5851.0,0.161621
3,2025-01-04,55430.0,11016.0,0.198737
4,2025-01-05,73687.0,16474.0,0.223567
5,2025-01-06,91838.0,22488.0,0.244866
6,2025-01-07,108654.0,29240.0,0.269111
7,2025-01-08,124099.0,33117.0,0.266860
8,2025-01-09,123399.0,32924.0,0.266809
9,2025-01-10,119905.0,33472.0,0.279154


In [123]:
carrier_daily = con.sql("""
SELECT
    FlightDate,
    Reporting_Airline,
    COUNT(*) AS daily_carrier_flights,
    SUM(significant_arrival_delay) AS daily_carrier_delays
FROM modeling_2025
GROUP BY
    FlightDate,
    Reporting_Airline
""")

In [125]:
carrier_7d_history = con.sql("""
SELECT
    FlightDate,
    Reporting_Airline,

    SUM(daily_carrier_flights) OVER (
        PARTITION BY Reporting_Airline
        ORDER BY FlightDate
        RANGE BETWEEN INTERVAL 7 DAY PRECEDING
                  AND INTERVAL 1 DAY PRECEDING
    ) AS prior_7d_carrier_flights,

    SUM(daily_carrier_delays) OVER (
        PARTITION BY Reporting_Airline
        ORDER BY FlightDate
        RANGE BETWEEN INTERVAL 7 DAY PRECEDING
                  AND INTERVAL 1 DAY PRECEDING
    ) AS prior_7d_carrier_delays

FROM carrier_daily
""")

In [126]:
carrier_7d_rates = con.sql("""
SELECT
    *,
    prior_7d_carrier_delays * 1.0
        / NULLIF(prior_7d_carrier_flights, 0)
        AS carrier_delay_rate_7d
FROM carrier_7d_history
""")

In [127]:
con.sql("""
SELECT *
FROM carrier_7d_rates
WHERE Reporting_Airline = 'AA'
ORDER BY FlightDate
LIMIT 12
""").df()

,FlightDate,Reporting_Airline,prior_7d_carrier_flights,prior_7d_carrier_delays,carrier_delay_rate_7d
0,2025-01-01,AA,NaN,NaN,NaN
1,2025-01-02,AA,2403.0,299.0,0.124428
2,2025-01-03,AA,5057.0,718.0,0.141981
3,2025-01-04,AA,7631.0,1264.0,0.165640
4,2025-01-05,AA,10001.0,1831.0,0.183082
5,2025-01-06,AA,12427.0,2812.0,0.226281
6,2025-01-07,AA,14778.0,3960.0,0.267966
7,2025-01-08,AA,16969.0,4319.0,0.254523
8,2025-01-09,AA,16792.0,4382.0,0.260958
9,2025-01-10,AA,15971.0,4388.0,0.274748


In [128]:
origin_daily = con.sql("""
SELECT
    FlightDate,
    Origin,
    COUNT(*) AS daily_origin_flights,
    SUM(significant_arrival_delay) AS daily_origin_delays
FROM modeling_2025
GROUP BY
    FlightDate,
    Origin
""")

origin_7d_history = con.sql("""
SELECT
    FlightDate,
    Origin,

    SUM(daily_origin_flights) OVER (
        PARTITION BY Origin
        ORDER BY FlightDate
        RANGE BETWEEN INTERVAL 7 DAY PRECEDING
                  AND INTERVAL 1 DAY PRECEDING
    ) AS prior_7d_origin_flights,

    SUM(daily_origin_delays) OVER (
        PARTITION BY Origin
        ORDER BY FlightDate
        RANGE BETWEEN INTERVAL 7 DAY PRECEDING
                  AND INTERVAL 1 DAY PRECEDING
    ) AS prior_7d_origin_delays

FROM origin_daily
""")

origin_7d_rates = con.sql("""
SELECT
    *,
    prior_7d_origin_delays * 1.0
        / NULLIF(prior_7d_origin_flights, 0)
        AS origin_delay_rate_7d
FROM origin_7d_history
""")

In [129]:
destination_daily = con.sql("""
SELECT
    FlightDate,
    Dest,
    COUNT(*) AS daily_destination_flights,
    SUM(significant_arrival_delay) AS daily_destination_delays
FROM modeling_2025
GROUP BY
    FlightDate,
    Dest
""")

destination_7d_history = con.sql("""
SELECT
    FlightDate,
    Dest,

    SUM(daily_destination_flights) OVER (
        PARTITION BY Dest
        ORDER BY FlightDate
        RANGE BETWEEN INTERVAL 7 DAY PRECEDING
                  AND INTERVAL 1 DAY PRECEDING
    ) AS prior_7d_destination_flights,

    SUM(daily_destination_delays) OVER (
        PARTITION BY Dest
        ORDER BY FlightDate
        RANGE BETWEEN INTERVAL 7 DAY PRECEDING
                  AND INTERVAL 1 DAY PRECEDING
    ) AS prior_7d_destination_delays

FROM destination_daily
""")

destination_7d_rates = con.sql("""
SELECT
    *,
    prior_7d_destination_delays * 1.0
        / NULLIF(prior_7d_destination_flights, 0)
        AS destination_delay_rate_7d
FROM destination_7d_history
""")

In [130]:
route_daily = con.sql("""
SELECT
    FlightDate,
    route,
    COUNT(*) AS daily_route_flights,
    SUM(significant_arrival_delay) AS daily_route_delays
FROM modeling_2025
GROUP BY
    FlightDate,
    route
""")

route_7d_history = con.sql("""
SELECT
    FlightDate,
    route,

    SUM(daily_route_flights) OVER (
        PARTITION BY route
        ORDER BY FlightDate
        RANGE BETWEEN INTERVAL 7 DAY PRECEDING
                  AND INTERVAL 1 DAY PRECEDING
    ) AS prior_7d_route_flights,

    SUM(daily_route_delays) OVER (
        PARTITION BY route
        ORDER BY FlightDate
        RANGE BETWEEN INTERVAL 7 DAY PRECEDING
                  AND INTERVAL 1 DAY PRECEDING
    ) AS prior_7d_route_delays

FROM route_daily
""")

route_7d_rates = con.sql("""
SELECT
    *,
    prior_7d_route_delays * 1.0
        / NULLIF(prior_7d_route_flights, 0)
        AS route_delay_rate_7d
FROM route_7d_history
""")

In [131]:
con.sql("""
SELECT *
FROM route_7d_rates
WHERE route = 'ATL_MCO'
ORDER BY FlightDate
LIMIT 12
""").df()

,FlightDate,route,prior_7d_route_flights,prior_7d_route_delays,route_delay_rate_7d
0,2025-01-01,ATL_MCO,NaN,NaN,NaN
1,2025-01-02,ATL_MCO,19.0,1.0,0.052632
2,2025-01-03,ATL_MCO,42.0,4.0,0.095238
3,2025-01-04,ATL_MCO,66.0,8.0,0.121212
4,2025-01-05,ATL_MCO,89.0,18.0,0.202247
5,2025-01-06,ATL_MCO,114.0,25.0,0.219298
6,2025-01-07,ATL_MCO,135.0,31.0,0.229630
7,2025-01-08,ATL_MCO,155.0,37.0,0.238710
8,2025-01-09,ATL_MCO,156.0,43.0,0.275641
9,2025-01-10,ATL_MCO,155.0,44.0,0.283871


## Joining Rolling 7-Day Features

The rolling 7-day features are joined back to each individual flight using the flight date and the relevant grouping key.

Each feature uses only information from the previous seven calendar days and excludes the current day, preserving the predeparture prediction constraint.

After joining, row counts are validated to ensure that feature engineering has not duplicated or removed flights.

In [132]:
modeling_with_7d = con.sql("""
SELECT
    m.*,

    s.prior_7d_system_flights,
    s.system_delay_rate_7d,

    c.prior_7d_carrier_flights,
    c.carrier_delay_rate_7d,

    o.prior_7d_origin_flights,
    o.origin_delay_rate_7d,

    d.prior_7d_destination_flights,
    d.destination_delay_rate_7d,

    r.prior_7d_route_flights,
    r.route_delay_rate_7d

FROM modeling_final_features m

LEFT JOIN system_7d_rates s
    ON m.FlightDate = s.FlightDate

LEFT JOIN carrier_7d_rates c
    ON m.FlightDate = c.FlightDate
   AND m.Reporting_Airline = c.Reporting_Airline

LEFT JOIN origin_7d_rates o
    ON m.FlightDate = o.FlightDate
   AND m.Origin = o.Origin

LEFT JOIN destination_7d_rates d
    ON m.FlightDate = d.FlightDate
   AND m.Dest = d.Dest

LEFT JOIN route_7d_rates r
    ON m.FlightDate = r.FlightDate
   AND m.route = r.route
""")

In [133]:
con.sql("""
SELECT
    (SELECT COUNT(*) FROM modeling_final_features) AS before_join,
    (SELECT COUNT(*) FROM modeling_with_7d) AS after_join
""").df()

,before_join,after_join
0,6879484,6879484


In [134]:
con.sql("""
SELECT
    COUNT(*) AS total_rows,

    SUM(CASE WHEN system_delay_rate_7d IS NULL THEN 1 ELSE 0 END)
        AS missing_system_7d,

    SUM(CASE WHEN carrier_delay_rate_7d IS NULL THEN 1 ELSE 0 END)
        AS missing_carrier_7d,

    SUM(CASE WHEN origin_delay_rate_7d IS NULL THEN 1 ELSE 0 END)
        AS missing_origin_7d,

    SUM(CASE WHEN destination_delay_rate_7d IS NULL THEN 1 ELSE 0 END)
        AS missing_destination_7d,

    SUM(CASE WHEN route_delay_rate_7d IS NULL THEN 1 ELSE 0 END)
        AS missing_route_7d

FROM modeling_with_7d
""").df()

,total_rows,missing_system_7d,missing_carrier_7d,missing_origin_7d,missing_destination_7d,missing_route_7d
0,6879484,16771.0,16771.0,16862.0,16862.0,21077.0


In [140]:
modeling_v2 = con.sql("""
SELECT
    *,

    -- Cumulative-history count fallbacks
    COALESCE(prior_carrier_flights, 0)
        AS prior_carrier_flights_ml,

    COALESCE(prior_origin_flights, 0)
        AS prior_origin_flights_ml,

    COALESCE(prior_destination_flights, 0)
        AS prior_destination_flights_ml,

    COALESCE(prior_route_flights, 0)
        AS prior_route_flights_ml,

    -- 7-day delay-rate fallbacks
    COALESCE(
        carrier_delay_rate_7d,
        system_delay_rate_7d
    ) AS carrier_delay_rate_7d_final,

    COALESCE(
        origin_delay_rate_7d,
        system_delay_rate_7d
    ) AS origin_delay_rate_7d_final,

    COALESCE(
        destination_delay_rate_7d,
        system_delay_rate_7d
    ) AS destination_delay_rate_7d_final,

    COALESCE(
        route_delay_rate_7d,
        origin_delay_rate_7d,
        destination_delay_rate_7d,
        carrier_delay_rate_7d,
        system_delay_rate_7d
    ) AS route_delay_rate_7d_final,

    -- 7-day history-count fallbacks
    COALESCE(prior_7d_carrier_flights, 0)
        AS prior_7d_carrier_flights_ml,

    COALESCE(prior_7d_origin_flights, 0)
        AS prior_7d_origin_flights_ml,

    COALESCE(prior_7d_destination_flights, 0)
        AS prior_7d_destination_flights_ml,

    COALESCE(prior_7d_route_flights, 0)
        AS prior_7d_route_flights_ml,

    -- 7-day missing-history indicators
    CASE WHEN carrier_delay_rate_7d IS NULL THEN 1 ELSE 0 END
        AS carrier_7d_missing,

    CASE WHEN origin_delay_rate_7d IS NULL THEN 1 ELSE 0 END
        AS origin_7d_missing,

    CASE WHEN destination_delay_rate_7d IS NULL THEN 1 ELSE 0 END
        AS destination_7d_missing,

    CASE WHEN route_delay_rate_7d IS NULL THEN 1 ELSE 0 END
        AS route_7d_missing

FROM modeling_with_7d

WHERE system_delay_rate_7d IS NOT NULL
""")

In [141]:
con.sql("""
SELECT
    COUNT(*) AS rows,
    MIN(FlightDate) AS first_date,
    MAX(FlightDate) AS last_date,

    SUM(CASE WHEN carrier_delay_rate_7d_final IS NULL THEN 1 ELSE 0 END)
        AS missing_carrier,

    SUM(CASE WHEN origin_delay_rate_7d_final IS NULL THEN 1 ELSE 0 END)
        AS missing_origin,

    SUM(CASE WHEN destination_delay_rate_7d_final IS NULL THEN 1 ELSE 0 END)
        AS missing_destination,

    SUM(CASE WHEN route_delay_rate_7d_final IS NULL THEN 1 ELSE 0 END)
        AS missing_route

FROM modeling_v2
""").df()

,rows,first_date,last_date,missing_carrier,missing_origin,missing_destination,missing_route
0,6862713,2025-01-02,2025-12-31,0.0,0.0,0.0,0.0


## Baseline v2 Model

Baseline v2 extends the original logistic regression model with leakage-safe 7-day rolling operational-history features.

The original cumulative historical features are retained to represent long-term performance, while the rolling features capture recent operational conditions.

To make the comparison with Baseline v1 meaningful, the modeling framework remains unchanged:

- Logistic regression
- 1,000,000 training observations
- January–September training period
- October–November validation period
- December final holdout period
- Identical preprocessing and evaluation metrics

The objective is to determine whether recent operational history provides incremental predictive value beyond cumulative historical performance.

In [144]:
rolling_7d_features = [
    "carrier_delay_rate_7d_final",
    "origin_delay_rate_7d_final",
    "destination_delay_rate_7d_final",
    "route_delay_rate_7d_final",
    "prior_7d_carrier_flights_ml",
    "prior_7d_origin_flights_ml",
    "prior_7d_destination_flights_ml",
    "prior_7d_route_flights_ml",
    "carrier_7d_missing",
    "origin_7d_missing",
    "destination_7d_missing",
    "route_7d_missing"
]

baseline_v2_features = baseline_v1_features + rolling_7d_features

print("Baseline v1 features:", len(baseline_v1_features))
print("New rolling features:", len(rolling_7d_features))
print("Baseline v2 features:", len(baseline_v2_features))

Baseline v1 features: 22
New rolling features: 12
Baseline v2 features: 34


In [145]:
categorical_features_v2 = categorical_features.copy()

numeric_features_v2 = [
    feature
    for feature in baseline_v2_features
    if feature not in categorical_features_v2
]

print("Categorical:", len(categorical_features_v2))
print("Numeric:", len(numeric_features_v2))
print("Total:", len(categorical_features_v2) + len(numeric_features_v2))

Categorical: 3
Numeric: 31
Total: 34


In [146]:
feature_sql_v2 = ", ".join(baseline_v2_features)

train_v2 = con.sql(f"""
SELECT
    {feature_sql_v2},
    significant_arrival_delay
FROM modeling_v2
WHERE Month BETWEEN 1 AND 9
""")

validation_v2 = con.sql(f"""
SELECT
    {feature_sql_v2},
    significant_arrival_delay
FROM modeling_v2
WHERE Month BETWEEN 10 AND 11
""")

In [147]:
con.sql("""
SELECT
    'train' AS split,
    COUNT(*) AS rows,
    ROUND(AVG(significant_arrival_delay) * 100, 2) AS delay_rate_pct
FROM train_v2

UNION ALL

SELECT
    'validation',
    COUNT(*),
    ROUND(AVG(significant_arrival_delay) * 100, 2)
FROM validation_v2
""").df()

: 

## Rebuild Modeling State After Kernel Restart

The Jupyter kernel was restarted during Baseline v2 development, which cleared the in-memory DuckDB relations and fitted Python objects.

Rather than rerunning the entire notebook and restoring unnecessary modeling objects, the required feature-engineering state is rebuilt from the processed 2025 flight data.

### Rebuilding Cumulative Historical Features

Leakage-safe cumulative historical features are reconstructed for:

- Reporting carrier
- Origin airport
- Destination airport
- Origin–destination route
- Overall flight system

For a flight occurring on date *t*, each historical feature uses observations only through *t − 1*. Current-day and future outcomes are excluded.

For each entity, two types of information are retained:

1. **Prior flight volume** — the number of flights observed before the prediction date.
2. **Historical delay rate** — the proportion of those prior flights that experienced a significant arrival delay of at least 15 minutes.

These cumulative features represent longer-term operational performance and will later be combined with the 7-day rolling features that capture more recent disruption conditions.

The rebuild intentionally avoids restoring the previous pandas training datasets and fitted Baseline v1 model in order to reduce memory usage while constructing Baseline v2.

In [1]:
import psutil

memory = psutil.virtual_memory()

print(f"Total RAM: {memory.total / 1024**3:.1f} GB")
print(f"Available RAM: {memory.available / 1024**3:.1f} GB")
print(f"Used RAM: {memory.used / 1024**3:.1f} GB")

Total RAM: 24.0 GB
Available RAM: 13.0 GB
Used RAM: 9.5 GB


In [2]:
from pathlib import Path
import duckdb
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

con = duckdb.connect()

In [3]:
all_flights_path = INTERIM_DIR / "flights_2025_*.parquet"

con.execute(
    f"""
    CREATE OR REPLACE VIEW flights_2025 AS
    SELECT *
    FROM read_parquet('{all_flights_path}')
    """
)

In [4]:
modeling_2025 = con.sql("""
SELECT
    Year,
    Quarter,
    Month,
    DayofMonth,
    DayOfWeek,
    FlightDate,
    Reporting_Airline,
    Origin,
    Dest,
    CRSDepTime,
    CRSArrTime,
    CRSElapsedTime,
    Distance,
    DistanceGroup,

    Origin || '_' || Dest AS route,

    CAST(
        FLOOR(CAST(CRSDepTime AS INTEGER) / 100)
        AS INTEGER
    ) AS scheduled_dep_hour,

    CAST(
        FLOOR(CAST(CRSArrTime AS INTEGER) / 100)
        AS INTEGER
    ) AS scheduled_arr_hour,

    CASE
        WHEN DayOfWeek IN (6, 7) THEN 1
        ELSE 0
    END AS is_weekend,

    ArrDelay,

    CASE
        WHEN ArrDelay >= 15 THEN 1
        ELSE 0
    END AS significant_arrival_delay

FROM flights_2025

WHERE Cancelled = 0
  AND Diverted = 0
  AND ArrDelay IS NOT NULL
""")

In [5]:
modeling_2025.aggregate("""
    COUNT(*) AS rows
""").df()

,rows
0,6879484


In [6]:
carrier_daily = con.sql("""
SELECT
    FlightDate,
    Reporting_Airline,
    COUNT(*) AS daily_flights,
    SUM(significant_arrival_delay) AS daily_delayed_flights
FROM modeling_2025
GROUP BY FlightDate, Reporting_Airline
""")

carrier_history_rates = con.sql("""
SELECT
    FlightDate,
    Reporting_Airline,
    SUM(daily_flights) OVER (
        PARTITION BY Reporting_Airline
        ORDER BY FlightDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS prior_carrier_flights,
    SUM(daily_delayed_flights) OVER (
        PARTITION BY Reporting_Airline
        ORDER BY FlightDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) * 1.0
    / NULLIF(
        SUM(daily_flights) OVER (
            PARTITION BY Reporting_Airline
            ORDER BY FlightDate
            ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
        ), 0
    ) AS carrier_historical_delay_rate
FROM carrier_daily
""")

In [7]:
origin_daily = con.sql("""
SELECT
    FlightDate,
    Origin,
    COUNT(*) AS daily_flights,
    SUM(significant_arrival_delay) AS daily_delayed_flights
FROM modeling_2025
GROUP BY FlightDate, Origin
""")

origin_history_rates = con.sql("""
SELECT
    FlightDate,
    Origin,
    SUM(daily_flights) OVER (
        PARTITION BY Origin
        ORDER BY FlightDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS prior_origin_flights,
    SUM(daily_delayed_flights) OVER (
        PARTITION BY Origin
        ORDER BY FlightDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) * 1.0
    / NULLIF(
        SUM(daily_flights) OVER (
            PARTITION BY Origin
            ORDER BY FlightDate
            ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
        ), 0
    ) AS origin_historical_delay_rate
FROM origin_daily
""")

In [8]:
destination_daily = con.sql("""
SELECT
    FlightDate,
    Dest,
    COUNT(*) AS daily_flights,
    SUM(significant_arrival_delay) AS daily_delayed_flights
FROM modeling_2025
GROUP BY FlightDate, Dest
""")

destination_history_rates = con.sql("""
SELECT
    FlightDate,
    Dest,
    SUM(daily_flights) OVER (
        PARTITION BY Dest
        ORDER BY FlightDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS prior_destination_flights,
    SUM(daily_delayed_flights) OVER (
        PARTITION BY Dest
        ORDER BY FlightDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) * 1.0
    / NULLIF(
        SUM(daily_flights) OVER (
            PARTITION BY Dest
            ORDER BY FlightDate
            ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
        ), 0
    ) AS destination_historical_delay_rate
FROM destination_daily
""")

In [9]:
route_daily = con.sql("""
SELECT
    FlightDate,
    route,
    COUNT(*) AS daily_flights,
    SUM(significant_arrival_delay) AS daily_delayed_flights
FROM modeling_2025
GROUP BY FlightDate, route
""")

route_history_rates = con.sql("""
SELECT
    FlightDate,
    route,
    SUM(daily_flights) OVER (
        PARTITION BY route
        ORDER BY FlightDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS prior_route_flights,
    SUM(daily_delayed_flights) OVER (
        PARTITION BY route
        ORDER BY FlightDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) * 1.0
    / NULLIF(
        SUM(daily_flights) OVER (
            PARTITION BY route
            ORDER BY FlightDate
            ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
        ), 0
    ) AS route_historical_delay_rate
FROM route_daily
""")

In [10]:
system_daily = con.sql("""
SELECT
    FlightDate,
    COUNT(*) AS daily_flights,
    SUM(significant_arrival_delay) AS daily_delayed_flights
FROM modeling_2025
GROUP BY FlightDate
""")

system_history_rates = con.sql("""
SELECT
    FlightDate,
    SUM(daily_flights) OVER (
        ORDER BY FlightDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS prior_system_flights,
    SUM(daily_delayed_flights) OVER (
        ORDER BY FlightDate
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) * 1.0
    / NULLIF(
        SUM(daily_flights) OVER (
            ORDER BY FlightDate
            ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
        ), 0
    ) AS system_historical_delay_rate
FROM system_daily
""")

In [11]:
carrier_history_rates.limit(5).df()

,FlightDate,Reporting_Airline,prior_carrier_flights,carrier_historical_delay_rate
0,2025-01-01,AS,NaN,NaN
1,2025-01-02,AS,623.0,0.104334
2,2025-01-03,AS,1314.0,0.149163
3,2025-01-04,AS,2004.0,0.228044
4,2025-01-05,AS,2634.0,0.241838


### Rebuilding 7-Day Rolling Historical Features

In addition to cumulative historical performance, Baseline v2 incorporates recent operational history using rolling 7-day windows.

For each prediction date, the rolling features summarize flight activity from the previous seven calendar days while explicitly excluding the current day. This preserves the leakage-safe design of the modeling pipeline.

Rolling features are calculated for:

- Overall flight system
- Reporting carrier
- Origin airport
- Destination airport
- Origin–destination route

These features capture short-term changes in disruption conditions that may not be reflected quickly in cumulative historical averages.

In [12]:
system_7d_rates = con.sql("""
SELECT
    FlightDate,

    SUM(daily_flights) OVER (
        ORDER BY FlightDate
        RANGE BETWEEN INTERVAL 7 DAY PRECEDING
                  AND INTERVAL 1 DAY PRECEDING
    ) AS prior_7d_system_flights,

    SUM(daily_delayed_flights) OVER (
        ORDER BY FlightDate
        RANGE BETWEEN INTERVAL 7 DAY PRECEDING
                  AND INTERVAL 1 DAY PRECEDING
    ) * 1.0
    / NULLIF(
        SUM(daily_flights) OVER (
            ORDER BY FlightDate
            RANGE BETWEEN INTERVAL 7 DAY PRECEDING
                      AND INTERVAL 1 DAY PRECEDING
        ), 0
    ) AS system_delay_rate_7d

FROM system_daily
""")

In [13]:
carrier_7d_rates = con.sql("""
SELECT
    FlightDate,
    Reporting_Airline,

    SUM(daily_flights) OVER (
        PARTITION BY Reporting_Airline
        ORDER BY FlightDate
        RANGE BETWEEN INTERVAL 7 DAY PRECEDING
                  AND INTERVAL 1 DAY PRECEDING
    ) AS prior_7d_carrier_flights,

    SUM(daily_delayed_flights) OVER (
        PARTITION BY Reporting_Airline
        ORDER BY FlightDate
        RANGE BETWEEN INTERVAL 7 DAY PRECEDING
                  AND INTERVAL 1 DAY PRECEDING
    ) * 1.0
    / NULLIF(
        SUM(daily_flights) OVER (
            PARTITION BY Reporting_Airline
            ORDER BY FlightDate
            RANGE BETWEEN INTERVAL 7 DAY PRECEDING
                      AND INTERVAL 1 DAY PRECEDING
        ), 0
    ) AS carrier_delay_rate_7d

FROM carrier_daily
""")

In [14]:
origin_7d_rates = con.sql("""
SELECT
    FlightDate,
    Origin,

    SUM(daily_flights) OVER (
        PARTITION BY Origin
        ORDER BY FlightDate
        RANGE BETWEEN INTERVAL 7 DAY PRECEDING
                  AND INTERVAL 1 DAY PRECEDING
    ) AS prior_7d_origin_flights,

    SUM(daily_delayed_flights) OVER (
        PARTITION BY Origin
        ORDER BY FlightDate
        RANGE BETWEEN INTERVAL 7 DAY PRECEDING
                  AND INTERVAL 1 DAY PRECEDING
    ) * 1.0
    / NULLIF(
        SUM(daily_flights) OVER (
            PARTITION BY Origin
            ORDER BY FlightDate
            RANGE BETWEEN INTERVAL 7 DAY PRECEDING
                      AND INTERVAL 1 DAY PRECEDING
        ), 0
    ) AS origin_delay_rate_7d

FROM origin_daily
""")

In [15]:
destination_7d_rates = con.sql("""
SELECT
    FlightDate,
    Dest,

    SUM(daily_flights) OVER (
        PARTITION BY Dest
        ORDER BY FlightDate
        RANGE BETWEEN INTERVAL 7 DAY PRECEDING
                  AND INTERVAL 1 DAY PRECEDING
    ) AS prior_7d_destination_flights,

    SUM(daily_delayed_flights) OVER (
        PARTITION BY Dest
        ORDER BY FlightDate
        RANGE BETWEEN INTERVAL 7 DAY PRECEDING
                  AND INTERVAL 1 DAY PRECEDING
    ) * 1.0
    / NULLIF(
        SUM(daily_flights) OVER (
            PARTITION BY Dest
            ORDER BY FlightDate
            RANGE BETWEEN INTERVAL 7 DAY PRECEDING
                      AND INTERVAL 1 DAY PRECEDING
        ), 0
    ) AS destination_delay_rate_7d

FROM destination_daily
""")

In [16]:
route_7d_rates = con.sql("""
SELECT
    FlightDate,
    route,

    SUM(daily_flights) OVER (
        PARTITION BY route
        ORDER BY FlightDate
        RANGE BETWEEN INTERVAL 7 DAY PRECEDING
                  AND INTERVAL 1 DAY PRECEDING
    ) AS prior_7d_route_flights,

    SUM(daily_delayed_flights) OVER (
        PARTITION BY route
        ORDER BY FlightDate
        RANGE BETWEEN INTERVAL 7 DAY PRECEDING
                  AND INTERVAL 1 DAY PRECEDING
    ) * 1.0
    / NULLIF(
        SUM(daily_flights) OVER (
            PARTITION BY route
            ORDER BY FlightDate
            RANGE BETWEEN INTERVAL 7 DAY PRECEDING
                      AND INTERVAL 1 DAY PRECEDING
        ), 0
    ) AS route_delay_rate_7d

FROM route_daily
""")

In [17]:
con.sql("""
SELECT *
FROM system_7d_rates
ORDER BY FlightDate
LIMIT 10
""").df()

,FlightDate,prior_7d_system_flights,system_delay_rate_7d
0,2025-01-01,NaN,NaN
1,2025-01-02,16771.0,0.153002
2,2025-01-03,36202.0,0.161621
3,2025-01-04,55430.0,0.198737
4,2025-01-05,73687.0,0.223567
5,2025-01-06,91838.0,0.244866
6,2025-01-07,108654.0,0.269111
7,2025-01-08,124099.0,0.266860
8,2025-01-09,123399.0,0.266809
9,2025-01-10,119905.0,0.279154


### Constructing the Baseline v2 Modeling Table

To reduce memory pressure, the Baseline v2 modeling table is constructed directly from the core 2025 flight relation and the required historical feature tables.

This avoids recreating the longer chain of intermediate relations used earlier in the notebook.

The resulting table combines:

- Scheduled predeparture flight features
- Cumulative carrier, airport, route, and system historical performance
- 7-day rolling carrier, airport, route, and system performance
- Historical observation counts
- Cold-start fallback indicators
- The significant-arrival-delay target

All historical features remain leakage-safe because they use only observations available before the prediction date.


In [18]:
modeling_v2 = con.sql("""
SELECT
    m.*,

    -- =========================
    -- CUMULATIVE HISTORY
    -- =========================

    c.prior_carrier_flights,
    c.carrier_historical_delay_rate,

    o.prior_origin_flights,
    o.origin_historical_delay_rate,

    d.prior_destination_flights,
    d.destination_historical_delay_rate,

    r.prior_route_flights,
    r.route_historical_delay_rate,

    s.prior_system_flights,
    s.system_historical_delay_rate,

    -- Cumulative rate fallbacks
    COALESCE(
        c.carrier_historical_delay_rate,
        s.system_historical_delay_rate
    ) AS carrier_delay_rate_final,

    COALESCE(
        o.origin_historical_delay_rate,
        s.system_historical_delay_rate
    ) AS origin_delay_rate_final,

    COALESCE(
        d.destination_historical_delay_rate,
        s.system_historical_delay_rate
    ) AS destination_delay_rate_final,

    COALESCE(
        r.route_historical_delay_rate,
        o.origin_historical_delay_rate,
        d.destination_historical_delay_rate,
        c.carrier_historical_delay_rate,
        s.system_historical_delay_rate
    ) AS route_delay_rate_final,

    -- Cumulative count fallbacks
    COALESCE(c.prior_carrier_flights, 0)
        AS prior_carrier_flights_ml,

    COALESCE(o.prior_origin_flights, 0)
        AS prior_origin_flights_ml,

    COALESCE(d.prior_destination_flights, 0)
        AS prior_destination_flights_ml,

    COALESCE(r.prior_route_flights, 0)
        AS prior_route_flights_ml,

    -- Cumulative cold-start indicators
    CASE WHEN c.carrier_historical_delay_rate IS NULL THEN 1 ELSE 0 END
        AS carrier_history_missing,

    CASE WHEN o.origin_historical_delay_rate IS NULL THEN 1 ELSE 0 END
        AS origin_history_missing,

    CASE WHEN d.destination_historical_delay_rate IS NULL THEN 1 ELSE 0 END
        AS destination_history_missing,

    CASE WHEN r.route_historical_delay_rate IS NULL THEN 1 ELSE 0 END
        AS route_history_missing,

    -- =========================
    -- 7-DAY HISTORY
    -- =========================

    s7.prior_7d_system_flights,
    s7.system_delay_rate_7d,

    c7.prior_7d_carrier_flights,
    c7.carrier_delay_rate_7d,

    o7.prior_7d_origin_flights,
    o7.origin_delay_rate_7d,

    d7.prior_7d_destination_flights,
    d7.destination_delay_rate_7d,

    r7.prior_7d_route_flights,
    r7.route_delay_rate_7d,

    -- 7-day rate fallbacks
    COALESCE(
        c7.carrier_delay_rate_7d,
        s7.system_delay_rate_7d
    ) AS carrier_delay_rate_7d_final,

    COALESCE(
        o7.origin_delay_rate_7d,
        s7.system_delay_rate_7d
    ) AS origin_delay_rate_7d_final,

    COALESCE(
        d7.destination_delay_rate_7d,
        s7.system_delay_rate_7d
    ) AS destination_delay_rate_7d_final,

    COALESCE(
        r7.route_delay_rate_7d,
        o7.origin_delay_rate_7d,
        d7.destination_delay_rate_7d,
        c7.carrier_delay_rate_7d,
        s7.system_delay_rate_7d
    ) AS route_delay_rate_7d_final,

    -- 7-day count fallbacks
    COALESCE(c7.prior_7d_carrier_flights, 0)
        AS prior_7d_carrier_flights_ml,

    COALESCE(o7.prior_7d_origin_flights, 0)
        AS prior_7d_origin_flights_ml,

    COALESCE(d7.prior_7d_destination_flights, 0)
        AS prior_7d_destination_flights_ml,

    COALESCE(r7.prior_7d_route_flights, 0)
        AS prior_7d_route_flights_ml,

    -- 7-day cold-start indicators
    CASE WHEN c7.carrier_delay_rate_7d IS NULL THEN 1 ELSE 0 END
        AS carrier_7d_missing,

    CASE WHEN o7.origin_delay_rate_7d IS NULL THEN 1 ELSE 0 END
        AS origin_7d_missing,

    CASE WHEN d7.destination_delay_rate_7d IS NULL THEN 1 ELSE 0 END
        AS destination_7d_missing,

    CASE WHEN r7.route_delay_rate_7d IS NULL THEN 1 ELSE 0 END
        AS route_7d_missing

FROM modeling_2025 m

LEFT JOIN carrier_history_rates c
    ON m.FlightDate = c.FlightDate
   AND m.Reporting_Airline = c.Reporting_Airline

LEFT JOIN origin_history_rates o
    ON m.FlightDate = o.FlightDate
   AND m.Origin = o.Origin

LEFT JOIN destination_history_rates d
    ON m.FlightDate = d.FlightDate
   AND m.Dest = d.Dest

LEFT JOIN route_history_rates r
    ON m.FlightDate = r.FlightDate
   AND m.route = r.route

LEFT JOIN system_history_rates s
    ON m.FlightDate = s.FlightDate

LEFT JOIN system_7d_rates s7
    ON m.FlightDate = s7.FlightDate

LEFT JOIN carrier_7d_rates c7
    ON m.FlightDate = c7.FlightDate
   AND m.Reporting_Airline = c7.Reporting_Airline

LEFT JOIN origin_7d_rates o7
    ON m.FlightDate = o7.FlightDate
   AND m.Origin = o7.Origin

LEFT JOIN destination_7d_rates d7
    ON m.FlightDate = d7.FlightDate
   AND m.Dest = d7.Dest

LEFT JOIN route_7d_rates r7
    ON m.FlightDate = r7.FlightDate
   AND m.route = r7.route

WHERE s7.system_delay_rate_7d IS NOT NULL
""")

In [19]:
con.sql("""
SELECT
    COUNT(*) AS rows,
    MIN(FlightDate) AS first_date,
    MAX(FlightDate) AS last_date
FROM modeling_v2
""").df()

,rows,first_date,last_date
0,6862713,2025-01-02,2025-12-31


In [20]:
con.sql("""
SELECT
    SUM(CASE WHEN carrier_delay_rate_final IS NULL THEN 1 ELSE 0 END)
        AS missing_carrier_cumulative,

    SUM(CASE WHEN route_delay_rate_final IS NULL THEN 1 ELSE 0 END)
        AS missing_route_cumulative,

    SUM(CASE WHEN carrier_delay_rate_7d_final IS NULL THEN 1 ELSE 0 END)
        AS missing_carrier_7d,

    SUM(CASE WHEN route_delay_rate_7d_final IS NULL THEN 1 ELSE 0 END)
        AS missing_route_7d

FROM modeling_v2
""").df()

,missing_carrier_cumulative,missing_route_cumulative,missing_carrier_7d,missing_route_7d
0,0.0,0.0,0.0,0.0


## Preparing Baseline v2 Training and Validation Data

Baseline v2 uses the same chronological evaluation framework as Baseline v1.

- Training period: January 2 through September 30, 2025
- Validation period: October 1 through November 30, 2025
- Final test period: December 2025 remains untouched

The Baseline v2 feature set retains the 22 original scheduled-flight and cumulative historical features and adds 12 leakage-safe 7-day rolling features.

To maintain a fair comparison with Baseline v1, the same logistic-regression framework and 1,000,000-row training sample size will be used.

In [21]:
categorical_features_v2 = [
    "Reporting_Airline",
    "Origin",
    "Dest"
]

rolling_7d_features = [
    "carrier_delay_rate_7d_final",
    "origin_delay_rate_7d_final",
    "destination_delay_rate_7d_final",
    "route_delay_rate_7d_final",
    "prior_7d_carrier_flights_ml",
    "prior_7d_origin_flights_ml",
    "prior_7d_destination_flights_ml",
    "prior_7d_route_flights_ml",
    "carrier_7d_missing",
    "origin_7d_missing",
    "destination_7d_missing",
    "route_7d_missing"
]

In [22]:
numeric_features_v1 = [
    "Month",
    "DayOfWeek",
    "scheduled_dep_hour",
    "scheduled_arr_hour",
    "is_weekend",
    "CRSElapsedTime",
    "Distance",

    "carrier_delay_rate_final",
    "origin_delay_rate_final",
    "destination_delay_rate_final",
    "route_delay_rate_final",

    "prior_carrier_flights_ml",
    "prior_origin_flights_ml",
    "prior_destination_flights_ml",
    "prior_route_flights_ml",

    "carrier_history_missing",
    "origin_history_missing",
    "destination_history_missing",
    "route_history_missing"
]

In [23]:
numeric_features_v2 = (
    numeric_features_v1
    + rolling_7d_features
)

baseline_v2_features = (
    categorical_features_v2
    + numeric_features_v2
)

print("Categorical features:", len(categorical_features_v2))
print("Numeric features:", len(numeric_features_v2))
print("Total features:", len(baseline_v2_features))

Categorical features: 3
Numeric features: 31
Total features: 34


In [24]:
feature_sql_v2 = ", ".join(baseline_v2_features)

train_v2 = con.sql(f"""
SELECT
    {feature_sql_v2},
    significant_arrival_delay
FROM modeling_v2
WHERE Month BETWEEN 1 AND 9
""")

validation_v2 = con.sql(f"""
SELECT
    {feature_sql_v2},
    significant_arrival_delay
FROM modeling_v2
WHERE Month BETWEEN 10 AND 11
""")

In [25]:
con.sql("""
SELECT
    'train' AS split,
    COUNT(*) AS rows,
    ROUND(
        AVG(significant_arrival_delay) * 100,
        2
    ) AS delay_rate_pct
FROM train_v2

UNION ALL

SELECT
    'validation',
    COUNT(*),
    ROUND(
        AVG(significant_arrival_delay) * 100,
        2
    )
FROM validation_v2
""").df()

,split,rows,delay_rate_pct
0,train,5133923,22.25
1,validation,1156866,20.44


## Baseline v2 Training

Baseline v2 retains the logistic-regression framework used in Baseline v1 but expands the feature set from 22 to 34 predictors by adding leakage-safe 7-day rolling operational-history features.

To keep the comparison fair:

- Training remains restricted to January–September 2025.
- Validation remains October–November 2025.
- December 2025 remains untouched.
- The training sample remains 1,000,000 observations.
- The same categorical preprocessing, numeric scaling, class balancing, solver, and optimization tolerance are used.

Any improvement in validation performance can therefore be attributed primarily to the addition of recent operational-history features rather than to a different modeling algorithm.

In [26]:
train_sample_v2 = con.sql(f"""
SELECT
    {feature_sql_v2},
    significant_arrival_delay
FROM modeling_v2
WHERE Month BETWEEN 1 AND 9
ORDER BY RANDOM()
LIMIT 1000000
""")

In [27]:
con.sql("""
SELECT
    COUNT(*) AS rows,
    ROUND(
        AVG(significant_arrival_delay) * 100,
        2
    ) AS delay_rate_pct
FROM train_sample_v2
""").df()

,rows,delay_rate_pct
0,1000000,22.32


In [28]:
train_df_v2 = train_sample_v2.df()

X_train_v2 = train_df_v2[baseline_v2_features]
y_train_v2 = train_df_v2["significant_arrival_delay"]

print("X_train_v2:", X_train_v2.shape)
print("y_train_v2:", y_train_v2.shape)

print(
    f"Memory: "
    f"{train_df_v2.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

X_train_v2: (1000000, 34)
y_train_v2: (1000000,)
Memory: 369.07 MB


In [29]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

In [30]:
preprocessor_v2 = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            ),
            categorical_features_v2
        ),
        (
            "numeric",
            StandardScaler(),
            numeric_features_v2
        )
    ]
)

In [31]:
baseline_model_v2 = Pipeline(
    steps=[
        ("preprocessor", preprocessor_v2),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                tol=1e-3,
                class_weight="balanced",
                solver="saga",
                random_state=42
            )
        )
    ]
)

In [32]:
baseline_model_v2.fit(
    X_train_v2,
    y_train_v2
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Reporting_Airline',
                                                   'Origin', 'Dest']),
                                                 ('numeric', StandardScaler(),
                                                  ['Month', 'DayOfWeek',
                                                   'scheduled_dep_hour',
                                                   'scheduled_arr_hour',
                                                   'is_weekend',
                                                   'CRSElapsedTime', 'Distance',
                                                   'carrier_delay_rate_final',
                                                   'origin_delay_rate_final',
                                                   'destina...
                                                   'destination_delay_rate_7d_final',
                                                   'route_delay_rate_7d_final',
                                                   'prior_7d_carrier_flights_ml',
                                                   'prior_7d_origin_flights_ml',
                                                   'prior_7d_destination_flights_ml',
                                                   'prior_7d_route_flights_ml',
                                                   'carrier_7d_missing',
                                                   'origin_7d_missing',
                                                   'destination_7d_missing', ...])])),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    random_state=42, solver='saga',
                                    tol=0.001))])

In [33]:
classifier_v2 = baseline_model_v2.named_steps["classifier"]

print(
    "Iterations used:",
    classifier_v2.n_iter_
)

Iterations used: [602]


### Baseline v2 Validation

Baseline v2 converged successfully after 602 iterations using the same logistic-regression configuration as Baseline v1.

The next step evaluates the model on the unchanged October–November validation period and compares the results directly with Baseline v1 to determine whether the 7-day rolling operational-history features add predictive value.

In [34]:
validation_df_v2 = validation_v2.df()

X_validation_v2 = validation_df_v2[baseline_v2_features]
y_validation_v2 = validation_df_v2["significant_arrival_delay"]

print(X_validation_v2.shape)

(1156866, 34)


In [37]:
y_val_pred_v2 = baseline_model_v2.predict(X_validation_v2)

y_val_prob_v2 = baseline_model_v2.predict_proba(
    X_validation_v2
)[:, 1]

In [38]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

In [39]:
validation_metrics_v2 = {
    "accuracy": accuracy_score(y_validation_v2, y_val_pred_v2),
    "precision": precision_score(y_validation_v2, y_val_pred_v2),
    "recall": recall_score(y_validation_v2, y_val_pred_v2),
    "f1": f1_score(y_validation_v2, y_val_pred_v2),
    "roc_auc": roc_auc_score(y_validation_v2, y_val_prob_v2),
    "pr_auc": average_precision_score(y_validation_v2, y_val_prob_v2)
}

validation_metrics_v2

{'accuracy': 0.6479047703018327,
 'precision': 0.30323788383959327,
 'recall': 0.5570359655022185,
 'f1': 0.3926991562760449,
 'roc_auc': np.float64(0.6591580801875917),
 'pr_auc': np.float64(0.31693132885157)}

In [42]:
validation_metrics_v2 = {
    "accuracy": accuracy_score(y_validation_v2, y_val_pred_v2),
    "precision": precision_score(y_validation_v2, y_val_pred_v2),
    "recall": recall_score(y_validation_v2, y_val_pred_v2),
    "f1": f1_score(y_validation_v2, y_val_pred_v2),
    "roc_auc": roc_auc_score(y_validation_v2, y_val_prob_v2),
    "pr_auc": average_precision_score(y_validation_v2, y_val_prob_v2)
}

validation_metrics_v2

{'accuracy': 0.6479047703018327,
 'precision': 0.30323788383959327,
 'recall': 0.5570359655022185,
 'f1': 0.3926991562760449,
 'roc_auc': np.float64(0.6591580801875917),
 'pr_auc': np.float64(0.31693132885157)}

In [41]:
confusion_matrix(
    y_validation_v2,
    y_val_pred_v2
)

array([[617844, 302601],
       [104726, 131695]])

### Baseline v2 Results

Adding leakage-safe 7-day rolling operational-history features improved the model's ability to distinguish higher-risk flights.

| Metric | Baseline v1 | Baseline v2 |
|---|---:|---:|
| Accuracy | 0.580 | 0.648 |
| Precision | 0.277 | 0.303 |
| Recall | 0.655 | 0.557 |
| F1 Score | 0.389 | 0.393 |
| ROC-AUC | 0.645 | 0.659 |
| PR-AUC | 0.299 | 0.317 |

The improvement in ROC-AUC and PR-AUC indicates that recent operational history provides incremental predictive information beyond cumulative historical performance.

At the default 0.50 classification threshold, Baseline v2 became more selective: precision and accuracy increased while recall decreased. Because threshold-dependent metrics can be adjusted later according to operational objectives, ROC-AUC and PR-AUC provide stronger evidence that the underlying ranking performance improved.

The improvement is meaningful but modest, suggesting that additional variations of historical-delay features alone are unlikely to produce a major increase in predictive performance. The next modeling stage should introduce new sources of information rather than continuing to optimize variations of the same historical signals.

# Phase 3 — NOAA Weather Integration

Baseline v2 demonstrated that recent operational history improves flight-delay prediction, but the improvement was modest. The next phase introduces an independent source of predictive information: historical weather observations.

Hourly weather data will be integrated with the BTS flight dataset using NOAA's Global Historical Climatology Network Hourly (GHCNh).

The weather pipeline will follow:

BTS Airport → NOAA Weather Station → Observation Timestamp → Weather Features → Flight Record

The initial weather feature set will focus on operationally relevant conditions available before scheduled departure, including:

- Temperature
- Dew point
- Relative humidity
- Wind speed
- Wind gust
- Visibility
- Precipitation
- Atmospheric pressure
- Sky conditions
- Present-weather indicators where available

Weather features will be constructed under the same leakage-control principle used throughout the project: information recorded after the prediction point will not be used.

December 2025 remains untouched as the final model holdout.

In [43]:
airport_summary = con.sql("""
SELECT
    airport,
    COUNT(*) AS flight_count
FROM (
    SELECT Origin AS airport
    FROM modeling_2025

    UNION ALL

    SELECT Dest AS airport
    FROM modeling_2025
)
GROUP BY airport
ORDER BY flight_count DESC
""").df()

print("Distinct airports:", len(airport_summary))

airport_summary.head(20)

Distinct airports: 352


,airport,flight_count
0,ORD,640515
1,DEN,627765
2,ATL,617859
3,DFW,611527
4,PHX,389639
5,CLT,383339
6,LAX,377447
7,LAS,364068
8,SEA,325942
9,MCO,315505


In [44]:
airport_summary["cumulative_flights"] = airport_summary["flight_count"].cumsum()

airport_summary["cumulative_pct"] = (
    airport_summary["cumulative_flights"]
    / airport_summary["flight_count"].sum()
    * 100
)

airport_summary.head(20)

,airport,flight_count,cumulative_flights,cumulative_pct
0,ORD,640515,640515,4.655255
1,DEN,627765,1268280,9.217843
2,ATL,617859,1886139,13.708434
3,DFW,611527,2497666,18.153004
4,PHX,389639,2887305,20.984895
5,CLT,383339,3270644,23.770998
6,LAX,377447,3648091,26.514278
7,LAS,364068,4012159,29.160319
8,SEA,325942,4338101,31.529261
9,MCO,315505,4653606,33.822348


In [45]:
for threshold in [80, 90, 95, 99]:
    airports_needed = (
        airport_summary["cumulative_pct"] < threshold
    ).sum() + 1

    print(
        f"Airports needed for {threshold}% of traffic: "
        f"{airports_needed}"
    )

Airports needed for 80% of traffic: 54
Airports needed for 90% of traffic: 92
Airports needed for 95% of traffic: 133
Airports needed for 99% of traffic: 232


## Transition to Weather Integration

The BTS-only modeling phase established two predictive benchmarks using temporally valid predeparture information.

Baseline v1 combined scheduled-flight characteristics with cumulative historical operational performance. Baseline v2 added leakage-safe 7-day rolling history and improved validation ROC-AUC from 0.645 to 0.659 and PR-AUC from 0.299 to 0.317.

Further transformations of historical delay performance are unlikely to provide a major increase in predictive power without introducing new information. The next phase therefore incorporates historical weather observations.

The 2025 BTS network contains 352 distinct airports. Airport traffic is concentrated but remains geographically broad:

- 54 airports account for 80% of flight endpoints
- 92 airports account for 90%
- 133 airports account for 95%
- 232 airports account for 99%

Weather integration will first be validated on high-volume airports before the station-matching and ingestion pipeline is generalized across the network.

Weather acquisition, airport-to-station mapping, temporal alignment, data-quality validation, and weather feature engineering are developed separately in `03_weather_integration.ipynb`.

December 2025 remains untouched as the final model holdout.
